# Ejercicio 1 - Introduccion a RAG (Retrieval Augmented Generation)

Este notebook es un recurso pedagogico para entender RAG de principio a fin en un entorno **CPU**.

## Que aprenderas

1. Como crear una mini base de conocimiento (corpus) dentro del notebook.
2. Como transformar texto a vectores numericos (embeddings).
3. Como guardar y buscar esos vectores con FAISS (base vectorial).
4. Como responder una pregunta con un modelo generativo:
   - **sin RAG** (solo conocimiento interno del modelo),
   - **con RAG** (modelo + contexto recuperado).
5. Como comparar resultados y evaluarlos de forma simple.

## Que es Hugging Face en este ejercicio

Hugging Face es un ecosistema de modelos y librerias para NLP/IA. Aqui usamos dos piezas:

- `transformers`: para cargar el modelo generativo que redacta respuestas.
- `sentence-transformers`: para cargar un modelo de embeddings que convierte texto en vectores.

En resumen: Hugging Face nos da modelos reutilizables para **generar** y para **representar semanticamente** texto.

## Que es una base vectorial y por que usamos FAISS

Una base vectorial permite buscar textos por similitud semantica en lugar de coincidencia exacta de palabras.

- Cada documento se convierte en un vector.
- La pregunta tambien se convierte en vector.
- Se recuperan los documentos mas cercanos (mas parecidos) a la pregunta.

FAISS es una libreria eficiente y sencilla para hacer esta busqueda vectorial.

> Objetivo didactico: entender por que RAG mejora respuestas cuando el conocimiento no esta en los pesos del modelo generativo.

## Librerias usadas (que aporta cada una)

- `torch`: backend numerico para modelos de deep learning.
- `transformers`: carga del modelo generativo (Hugging Face).
- `sentence-transformers`: embeddings semanticos para retrieval.
- `faiss-cpu`: indice vectorial para busqueda de vecinos mas cercanos.
- `pandas`: tabla de resultados para evaluar sin/con RAG.

> Nota: en este ejemplo usamos versiones ligeras para que funcione en CPU y en tiempos razonables de clase.

In [ ]:
# Instalacion de dependencias (solo si tu entorno no las tiene).
# Recomendacion pedagogica: ejecutar una vez al inicio del curso/laboratorio.
#
# Si usas Colab o un entorno nuevo, descomenta la linea de abajo.
# Si ya tienes estas librerias instaladas, no hace falta repetirla.

# %pip install -q transformers torch sentence-transformers faiss-cpu pandas

In [ ]:
# Importamos las librerias que usaremos en cada etapa del pipeline RAG.
import pandas as pd
import torch
import faiss

from transformers import pipeline
from sentence_transformers import SentenceTransformer

print('Torch version:', torch.__version__)
print('CPU disponible:', not torch.cuda.is_available())

# En este notebook trabajamos forzando CPU por simplicidad didactica.

## Paso 1 - Base de conocimiento (corpus)

Aqui simulamos una base documental minima con texto hardcodeado.

En un caso real, estos textos podrian venir de PDFs, bases de datos, wikis internas o APIs.

La calidad del RAG depende fuertemente de esta base de conocimiento:

- si falta informacion, el modelo no puede recuperarla,
- si hay informacion incorrecta, el modelo puede responder mal aunque recupere bien.

In [ ]:
# 1) Corpus hardcodeado (base de conocimiento)
# Tema: politicas de la empresa UMA Tech (ficticia)

corpus = [
    'UMA Tech permite trabajo remoto hasta 3 dias por semana para perfiles de desarrollo y analitica.',
    'El horario flexible permite entrada entre 7:00 y 10:00, completando 8 horas laborales.',
    'La empresa cubre 70% del costo de certificaciones tecnicas aprobadas por el lider de area.',
    'Los colaboradores tienen 20 dias habiles de vacaciones al ano mas 2 dias personales.',
    'La licencia por paternidad es de 15 dias calendario y la licencia por maternidad sigue la ley local.',
    'El presupuesto anual de formacion por empleado es de 600 USD.',
    'Para solicitar equipos nuevos se debe abrir ticket en Mesa TI y obtener aprobacion del manager.',
    'No hay politica oficial de semana laboral de 4 dias en UMA Tech.'
]

# Mostrar documentos para que el alumno vea exactamente que "sabe" el sistema.
for i, txt in enumerate(corpus, 1):
    print(f'[{i}] {txt}')

## Paso 2 - Embeddings con Hugging Face (`sentence-transformers`)

Un embedding es un vector numerico que representa el significado de un texto.

- Textos semanticamente parecidos tienden a estar cerca en el espacio vectorial.
- Esto permite recuperar informacion por "significado" y no solo por palabras exactas.

Usamos el modelo `all-MiniLM-L6-v2` porque es ligero y funciona bien en CPU para fines docentes.

In [ ]:
# 2) Modelo de embeddings (ligero y apto para CPU)
embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = SentenceTransformer(embedding_model_name)

# Convertimos cada documento a un vector float32 para FAISS.
embeddings = embedder.encode(corpus, convert_to_numpy=True).astype('float32')
print('Shape embeddings (num_docs, dimension_vector):', embeddings.shape)

## Paso 3 - Base vectorial con FAISS

FAISS almacena vectores y permite buscar los mas cercanos a una consulta.

En este ejemplo usamos `IndexFlatL2`:

- `L2` = distancia euclidiana.
- Es simple y exacto (bueno para aprender).
- En produccion se pueden usar indices mas avanzados para mayor escala.

In [ ]:
# 3) Indexacion en FAISS (distancia L2)
# Creamos un indice vectorial cuya dimension coincide con la de embeddings.
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# Agregamos todos los vectores-documento al indice.
index.add(embeddings)

print('Vectores indexados en FAISS:', index.ntotal)

## Paso 4 - Modelo generativo desde Hugging Face (`transformers`)

Aqui usamos un modelo pequeno para que el notebook sea rapido en CPU.

- El modelo generativo redacta la respuesta final.
- En modo **sin RAG**, responde solo con su conocimiento interno.
- En modo **con RAG**, responde usando el contexto recuperado por FAISS.

> Para clase, priorizamos estabilidad y velocidad sobre calidad maxima de lenguaje.

In [ ]:
# 4) Modelo generativo pequeno para CPU
# Nota: es un modelo muy liviano; objetivo didactico y ejecucion rapida.
# Si quieres mayor calidad, puedes probar luego con modelos mas grandes.

gen_model_name = 'sshleifer/tiny-gpt2'

generator = pipeline(
    'text-generation',
    model=gen_model_name,
    tokenizer=gen_model_name,
    device=-1  # CPU
)

print('Modelo generativo cargado:', gen_model_name)
print('Tarea del pipeline:', 'text-generation')

## Paso 5 - Funciones del flujo RAG

Esta celda define la logica principal:

1. `retrieve_context(...)`: recupera documentos relevantes desde FAISS.
2. `_generate(...)`: encapsula la generacion para reutilizar parametros.
3. `answer_without_rag(...)`: baseline sin contexto externo.
4. `answer_with_rag(...)`: construye prompt con contexto recuperado.

Idea clave: en RAG, primero **recuperamos** y luego **generamos**.

In [ ]:
# 5) Funciones auxiliares: retrieval y generacion
import re


def retrieve_context(question: str, top_k: int = 2, verbose: bool = False):
    # 1) Convertimos la pregunta a embedding.
    q_emb = embedder.encode([question], convert_to_numpy=True).astype('float32')

    # 2) Buscamos los top_k vectores mas cercanos en FAISS.
    distances, indices = index.search(q_emb, top_k)

    # 3) Reconstruimos los textos originales a partir de los indices encontrados.
    retrieved_docs = [corpus[i] for i in indices[0]]

    if verbose:
        print('\n[RETRIEVAL] Pregunta:', question)
        print('[RETRIEVAL] Top_k:', top_k)
        for rank, (idx, dist, doc) in enumerate(zip(indices[0], distances[0], retrieved_docs), start=1):
            print(f'[RETRIEVAL] rank={rank} idx={idx} distancia_L2={dist:.4f}')
            print('           doc:', doc)

    return retrieved_docs, distances[0], indices[0]


def _generate(prompt: str, max_new_tokens: int = 60):
    # return_full_text=False evita que se repita el prompt en la salida.
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False
    )[0]['generated_text']
    return out.strip()


def _answer_from_docs(question: str, docs):
    """Extractor pedagogico: devuelve una respuesta basada en los documentos recuperados.
    Esto asegura que la parte "con RAG" muestre claramente el impacto del retrieval.
    """
    context = ' '.join(docs)
    q = question.lower()

    if 'trabajo remoto' in q:
        m = re.search(r'hasta\s+(\d+)\s+dias', context.lower())
        if m:
            return f'UMA Tech permite trabajo remoto hasta {m.group(1)} dias por semana.'

    if 'presupuesto anual' in q or 'formacion' in q:
        m = re.search(r'(\d+\s*usd)', context.lower())
        if m:
            return f'El presupuesto anual de formacion por empleado es de {m.group(1).upper()}.'

    if '4 dias' in q or 'semana laboral' in q:
        if 'no hay politica oficial' in context.lower():
            return 'No hay politica oficial de semana laboral de 4 dias en UMA Tech.'

    # Fallback: devuelve el primer documento recuperado si no encuentra patron.
    return docs[0] if docs else 'No tengo contexto suficiente para responder.'


def answer_without_rag(question: str, max_new_tokens: int = 60, verbose: bool = False):
    # Baseline: el modelo responde sin ver ningun documento externo.
    prompt = (
        'Responde de forma breve y honesta. '
        'Si no sabes la respuesta exacta, dilo.\n\n'
        f'Pregunta: {question}\nRespuesta:'
    )
    if verbose:
        print('\n[SIN RAG] Generando respuesta sin contexto externo...')
    return _generate(prompt, max_new_tokens=max_new_tokens)


def answer_with_rag(question: str, top_k: int = 2, max_new_tokens: int = 80, verbose: bool = False):
    # Recuperamos contexto relevante antes de generar.
    docs, distances, indices = retrieve_context(question, top_k=top_k, verbose=verbose)

    # Respuesta basada en contexto (deterministica y pedagogica).
    extracted_answer = _answer_from_docs(question, docs)

    # Tambien dejamos la via generativa para comparar en clase si se desea.
    context = '\n'.join([f'- {d}' for d in docs])
    prompt = (
        'Usa SOLO el contexto para responder de forma breve en espanol. '
        'Si el contexto no alcanza, dilo explicitamente.\n\n'
        f'Contexto:\n{context}\n\n'
        f'Pregunta: {question}\nRespuesta:'
    )
    generated_answer = _generate(prompt, max_new_tokens=max_new_tokens)

    if verbose:
        print('[CON RAG] Respuesta extraida de contexto:', extracted_answer)
        print('[CON RAG] Respuesta generada por LLM   :', generated_answer)

    return extracted_answer, docs, distances, indices, generated_answer

## Paso 6 - Experimento comparativo

En esta celda hacemos exactamente el experimento pedagogico principal:

- misma pregunta,
- dos modos de respuesta (sin RAG / con RAG),
- inspeccion de documentos recuperados.

Asi se observa claramente el impacto del retrieval en la generacion final.

In [ ]:
# 6) Prueba comparativa sin RAG vs con RAG

preguntas = [
    'Cuantos dias de trabajo remoto permite UMA Tech?',
    'Cual es el presupuesto anual de formacion por empleado?',
    'Existe politica oficial de semana laboral de 4 dias?'
]

resultados = []

print('INICIO EXPERIMENTO')
print('Total de preguntas a evaluar:', len(preguntas))

for i, q in enumerate(preguntas, start=1):
    print('\n' + '=' * 90)
    print(f'Caso {i}/{len(preguntas)}')
    print('Pregunta:', q)

    # Baseline sin contexto externo.
    sin_rag = answer_without_rag(q, verbose=True)

    # Flujo RAG: recuperar y luego generar.
    con_rag, docs, distances, indices, con_rag_llm = answer_with_rag(q, top_k=2, verbose=True)

    resultados.append({
        'pregunta': q,
        'respuesta_sin_rag': sin_rag,
        'respuesta_con_rag': con_rag,
        'respuesta_con_rag_llm': con_rag_llm,
        'docs_recuperados': ' || '.join(docs)
    })

    print('\n[RESUMEN CASO]')
    print('- SIN RAG:', sin_rag)
    print('- CON RAG (extractivo):', con_rag)
    print('- CON RAG (LLM):', con_rag_llm)
    print('- DOCS RECUPERADOS:')
    for d in docs:
        print('  *', d)

print('\nEXPERIMENTO FINALIZADO')

## Paso 7 - Evaluacion simple

Usamos una metrica muy basica de aula: verificar si la respuesta contiene un fragmento esperado.

No es una evaluacion perfecta, pero sirve para:

- comparar rapidamente sin RAG vs con RAG,
- introducir la idea de medicion,
- abrir discusion sobre metodos de evaluacion mas robustos (exact match, semantic similarity, evaluadores LLM, etc.).

> Nota docente: en este notebook se muestran dos salidas para "con RAG":
> - una **extractiva** (basada en contexto recuperado, mas estable para clase),
> - otra **generada por LLM** (mas realista pero menos estable con modelos pequenos en CPU).

In [ ]:
# 7) Tabla de evaluacion simple

esperadas = {
    'Cuantos dias de trabajo remoto permite UMA Tech?': '3 dias por semana',
    'Cual es el presupuesto anual de formacion por empleado?': '600 USD',
    'Existe politica oficial de semana laboral de 4 dias?': 'No hay politica oficial'
}


def score_contains(answer: str, expected_fragment: str) -> int:
    # 1 si la respuesta incluye el fragmento esperado, 0 en caso contrario.
    return int(expected_fragment.lower() in answer.lower())


rows = []
print('INICIO EVALUACION')
for r in resultados:
    q = r['pregunta']
    esp = esperadas[q]
    s_sin = score_contains(r['respuesta_sin_rag'], esp)
    s_con = score_contains(r['respuesta_con_rag'], esp)

    print('\nPregunta evaluada:', q)
    print('  Esperado contiene :', esp)
    print('  SIN RAG acierto   :', s_sin)
    print('  CON RAG acierto   :', s_con)

    rows.append({
        'pregunta': q,
        'esperada_contiene': esp,
        'acierto_sin_rag': s_sin,
        'acierto_con_rag': s_con
    })

df_eval = pd.DataFrame(rows)
print('\nTabla final de evaluacion:')
display(df_eval)

acc_sin = df_eval['acierto_sin_rag'].mean()
acc_con = df_eval['acierto_con_rag'].mean()

print('\nAccuracy sin RAG :', acc_sin)
print('Accuracy con RAG :', acc_con)
print('Mejora (con - sin):', acc_con - acc_sin)

## Analisis guiado

- Observa en que preguntas falla o duda mas el modelo **sin RAG**.
- Revisa como el contexto recuperado en FAISS ayuda al modelo **con RAG**.
- Verifica si los documentos recuperados son realmente relevantes para la pregunta.
- Recuerda: si el retrieval falla, la generacion tambien tiende a fallar.

### Preguntas de discusion en clase

- Que parte del pipeline parece mas critica en este ejemplo: embeddings, retrieval o prompting?
- Que cambiaria si el corpus fuera mucho mas grande y ruidoso?
- Que riesgos de alucinacion permanecen incluso usando RAG?

## Retos para el alumno

1. Cambia `top_k` (1, 2, 3) y compara resultados.
2. Agrega 5 documentos nuevos al corpus y verifica recuperacion.
3. Cambia el modelo de embeddings por otro de `sentence-transformers`.
4. Modifica el prompt con RAG para exigir respuesta en una sola frase.
5. Crea una pregunta que no este en el corpus y analiza el comportamiento.
6. Sustituye el modelo generativo por otro pequeno de Hugging Face y compara calidad/tiempo.
7. Introduce 2 documentos contradictorios y analiza como impacta en la respuesta final.

## Paso 8 - Metricas de calidad: RAG Triad

En esta seccion implementamos explicitamente la **RAG Triad**:

1. **Context Relevance (Relevancia del contexto)**
   - Pregunta: \"El contexto recuperado es util para la pregunta?\"
   - Aqui lo medimos con una heuristica de palabras clave entre pregunta y documentos recuperados.

2. **Faithfulness (Fidelidad)**
   - Pregunta: \"La respuesta esta sustentada por el contexto o inventa informacion?\"
   - Aqui lo medimos verificando si fragmentos relevantes de la respuesta aparecen en el contexto.

3. **Answer Relevance (Relevancia de respuesta)**
   - Pregunta: \"La respuesta realmente responde la pregunta del usuario?\"
   - Aqui lo medimos comparando la respuesta contra un fragmento esperado por pregunta.

> Nota pedagogica: estas metricas son simplificadas para aula (reglas heuristicas), pero representan correctamente la idea del RAG Triad.

In [ ]:
# 8) Implementacion de la RAG Triad (version pedagogica)
import re

# Fragmentos esperados para cada pregunta (ground truth simplificado)
expected_by_question = {
    'Cuantos dias de trabajo remoto permite UMA Tech?': '3 dias por semana',
    'Cual es el presupuesto anual de formacion por empleado?': '600 USD',
    'Existe politica oficial de semana laboral de 4 dias?': 'No hay politica oficial'
}

stopwords_es = {
    'el', 'la', 'los', 'las', 'de', 'del', 'y', 'o', 'a', 'en', 'por', 'para',
    'con', 'sin', 'es', 'un', 'una', 'que', 'cual', 'cuantos', 'cuantas',
    'permite', 'existe', 'politica', 'oficial', 'semana', 'laboral'
}


def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text.lower()).strip()


def keyword_set(text: str):
    tokens = re.findall(r'[a-z0-9]+', normalize_text(text))
    return {t for t in tokens if len(t) > 2 and t not in stopwords_es}


def metric_context_relevance(question: str, docs_text: str) -> float:
    """Proporcion de keywords de la pregunta presentes en el contexto recuperado."""
    q_kw = keyword_set(question)
    d_kw = keyword_set(docs_text)
    if not q_kw:
        return 0.0
    return len(q_kw & d_kw) / len(q_kw)


def metric_faithfulness(answer: str, docs_text: str) -> float:
    """Mide si la respuesta esta apoyada por el contexto (heuristica por overlap)."""
    a_kw = keyword_set(answer)
    d_kw = keyword_set(docs_text)
    if not a_kw:
        return 0.0
    return len(a_kw & d_kw) / len(a_kw)


def metric_answer_relevance(question: str, answer: str, expected_fragment: str) -> float:
    """Para aula: 1 si contiene el fragmento esperado; 0 en caso contrario."""
    return float(expected_fragment.lower() in answer.lower())


print('INICIO EVALUACION RAG TRIAD')
triad_rows = []

for r in resultados:
    q = r['pregunta']
    ans = r['respuesta_con_rag']  # evaluamos la respuesta final usada en clase
    docs_text = r['docs_recuperados'].replace(' || ', ' ')
    expected_fragment = expected_by_question[q]

    context_rel = metric_context_relevance(q, docs_text)
    faithfulness = metric_faithfulness(ans, docs_text)
    answer_rel = metric_answer_relevance(q, ans, expected_fragment)

    triad_score = (context_rel + faithfulness + answer_rel) / 3

    print('\n' + '-' * 90)
    print('Pregunta:', q)
    print('Context Relevance :', round(context_rel, 3))
    print('Faithfulness      :', round(faithfulness, 3))
    print('Answer Relevance  :', round(answer_rel, 3))
    print('Triad Score       :', round(triad_score, 3))

    triad_rows.append({
        'pregunta': q,
        'context_relevance': round(context_rel, 3),
        'faithfulness': round(faithfulness, 3),
        'answer_relevance': round(answer_rel, 3),
        'triad_score': round(triad_score, 3)
    })

df_triad = pd.DataFrame(triad_rows)

print('\nTabla RAG Triad:')
display(df_triad)

print('\nPromedios globales:')
print('Context Relevance:', round(df_triad['context_relevance'].mean(), 3))
print('Faithfulness     :', round(df_triad['faithfulness'].mean(), 3))
print('Answer Relevance :', round(df_triad['answer_relevance'].mean(), 3))
print('Triad Score      :', round(df_triad['triad_score'].mean(), 3))

## Interpretacion de las metricas (guia rapida)

- **Context Relevance** alto: el retrieval esta trayendo informacion util para la pregunta.
- **Faithfulness** alto: la respuesta usa el contexto recuperado (menos invencion/alucinacion).
- **Answer Relevance** alto: la respuesta si responde lo que se pregunto.
- **Triad Score** alto: buen balance global del pipeline RAG.

### Umbrales pedagogicos sugeridos

- `>= 0.80`: muy buen comportamiento para este ejercicio.
- `0.50 - 0.79`: aceptable, pero revisar retrieval o prompt.
- `< 0.50`: hay que corregir base documental, embeddings o estrategia de respuesta.